In [1]:
def determine_basin_name(site_df, grills_basins_df, grills_full, basin_col="RIVER_BASIN", grills_full_col=["GRILSS RID", "Reservoir"]):
    """
    Accepts a sites dataframe and returns the basin name using the grills_basins geojson file by spatial matching
    """
    sites_copy = site_df.copy()
    basins = grills_basins_df[[basin_col, grills_basins_df.geometry.name]].copy()
    geom_col = grills_full.geometry.name  # usually "geometry"
    cols = list(grills_full_col) if isinstance(grills_full_col, (list, tuple)) else [grills_full_col]
    cols = cols + [geom_col]
    grills_dams_grillsIDmap = grills_full[cols].copy()
    
    #buffer grills_dams_grillsIDmap to 0.01 degrees
    grills_dams_grillsIDmap["geometry"] = grills_dams_grillsIDmap["geometry"].buffer(0.02)
    ##some processing to remove conflicting columns required for gpd.sjoin fucntion
    for bad in ("index_left", "index_right"):
        if bad in sites_copy.columns:
            sites_copy = sites_copy.rename(columns={bad: f"site_{bad}"})
        if bad in basins.columns:
            basins = basins.rename(columns={bad: f"basin_{bad}"})
        if bad in grills_dams_grillsIDmap.columns:
            grills_dams_grillsIDmap = grills_dams_grillsIDmap.rename(columns={bad: f"grills_{bad}"})
    ##
    # converting sites_copy to same crs as grills    
    if sites_copy.crs != basins.crs:
        sites_copy = sites_copy.to_crs(basins.crs)
    

    #spatial join to get the RIVER_BASIN column of grills into sites df
    joined_basins = gpd.sjoin(sites_copy, basins, how="left", predicate="within")    
    #remove index_left and index_right columns
    joined_basins = joined_basins.drop(columns=["index_right"])
    joined_grillsID = gpd.sjoin(joined_basins, grills_dams_grillsIDmap, how="left", predicate="intersects")

    #drop null from GRILSS RID column
    joined_grillsID = joined_grillsID.dropna(subset=["GRILSS RID"]).copy()
    #convert GRILSS RID to int
    joined_grillsID['GRILSS RID'] = joined_grillsID['GRILSS RID'].astype(int)
    return joined_grillsID
    

In [2]:
def compute_monthly_metrics(df, method='sum', date_col=None, freq='M'):
    """
    Aggregate to monthly frequency using resample.

    df        : DataFrame or GeoDataFrame. If date_col is None, index must be DatetimeIndex.
    method    : aggregation (e.g., 'sum', 'mean', np.sum, np.mean, dict of col->agg, etc.)
    date_col  : name of the date column to set as index (optional).
    freq      : 'M' (month end) or 'MS' (month start).
    """
    # Ensure datetime index
    if date_col is not None:
        df = df.set_index(pd.to_datetime(df[date_col]))
    elif not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("Provide date_col or set a DatetimeIndex before resampling.")

    out = df.resample(freq).agg(method)

    # Make the index nice: 1981-07, 1981-08, ...
    out.index = out.index.to_period('M')
    out.index.name = 'month'
    return out



In [3]:
def compute_correl (df1,df2, col1, col2):
    #drop any nans
    df1 = df1.dropna()
    df2 = df2.dropna()
    return(df1[col1].corr(df2[col2]))

In [4]:
def monthly_zscore(series):
    """
    Compute site-specific, calendar-month Z-score.
    """
    return (
        series
        - series.groupby(series.index.month).transform('mean')
    ) / (
        series.groupby(series.index.month).transform('std') + 1e-6
    )
    




In [5]:
def robust_trend(y):
    if len(y) < 6:
        return np.nan
    x = np.arange(len(y))
    slope, _, _, _ = theilslopes(y.values, x)
    return slope


In [6]:
def identify_growing_season_months(
    gpp_series,
    quantile_thr=0.5
):
    """
    Identify growing-season months based on climatological GPP.

    Parameters
    ----------
    gpp_series : pandas.Series
        Monthly GPP time series indexed by datetime
    quantile_thr : float
        Quantile threshold (default = median)

    Returns
    -------
    growing_months : list[int]
        List of calendar months considered growing season
    gpp_climatology : pandas.Series
        Mean GPP for each calendar month
    """

    gpp_climatology = (
        gpp_series
        .groupby(gpp_series.index.month)
        .mean()
    )

    thr = gpp_climatology.quantile(quantile_thr)
    growing_months = gpp_climatology[gpp_climatology >= thr].index.tolist()

    return growing_months, gpp_climatology


In [7]:
def compute_gpp_residuals_precip_control(
    df,
    growing_months
):
    """
    Compute GPP residuals after removing precipitation influence,
    using only growing-season months.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain 'modis_gpp' and 'precip'
    growing_months : list[int]
        Calendar months considered growing season

    Returns
    -------
    df_out : pandas.DataFrame
        Copy of df with column 'GPP_res'
    model : fitted LinearRegression
    """

    df = df.copy()

    df['is_growing'] = df.index.month.isin(growing_months)

    mask = (
        df['is_growing'] &
        df['modis_gpp'].notna() &
        df['precip'].notna()
    )

    if mask.sum() < 12:
        df['GPP_res'] = np.nan
        return df, None

    X = df.loc[mask, ['precip']]
    y = df.loc[mask, 'modis_gpp']

    model = LinearRegression().fit(X, y)

    df['GPP_res'] = np.nan
    df.loc[mask, 'GPP_res'] = y - model.predict(X)

    return df, model


In [8]:
def compute_sediment_trend_benefit(
    df,
    flood_event_table,
    pre_months=24,
    post_months=24,
    buffer_months=3,
    min_valid_months=12,
    beta_cap=0.05
):
    """
    Metric 3B (Trend-based):
    Change in long-term GPP trend after dam-attributed flooding,
    with post-flood window ending at next flood.

    Returns
    -------
    {
      'B_sediment_trend': float or NaN,
      'N_dam_events': int,
      'N_valid_events': int,
      'event_table': DataFrame
    }
    """

    if df is None or df.empty or 'Z_GPP' not in df.columns:
        return {
            'B_sediment_trend': np.nan,
            'N_dam_events': 0,
            'N_valid_events': 0,
            'event_table': pd.DataFrame()
        }

    flood_event_table = flood_event_table.sort_values('start_date')
    dam_events = flood_event_table[flood_event_table['dam_attributed']]

    events = []

    flood_starts = pd.to_datetime(flood_event_table['start_date']).values

    for i, (_, row) in enumerate(dam_events.iterrows()):
        t0 = pd.to_datetime(row['start_date'])

        # Find next flood after this one
        later_floods = flood_starts[flood_starts > t0]
        if len(later_floods) > 0:
            t_next = pd.to_datetime(later_floods.min())
            post_end = t_next - pd.DateOffset(months=1)
        else:
            post_end = t0 + pd.DateOffset(months=buffer_months + post_months)

        post_start = t0 + pd.DateOffset(months=buffer_months)

        pre = df.loc[
            t0 - pd.DateOffset(months=pre_months) :
            t0 - pd.DateOffset(months=1),
            'Z_GPP'
        ].dropna()

        post = df.loc[
            post_start :
            post_end,
            'Z_GPP'
        ].dropna()

        mean_pre = pre.mean()
        mean_post = post.mean()

        has_recovered_level = mean_post >= mean_pre

        if len(pre) < min_valid_months or len(post) < min_valid_months:
            continue

        beta_pre = robust_trend(pre)
        beta_post = robust_trend(post)

        if np.isnan(beta_pre) or np.isnan(beta_post):
            continue

        delta_beta = beta_post - beta_pre
        # score = np.clip(delta_beta / beta_cap, 0, 1)
        if (delta_beta > 0) and has_recovered_level:
            score = np.clip(delta_beta / beta_cap, 0, 1)
        else:
            score = 0.0


        events.append({
            'event_start': t0,
            'next_flood': t_next if len(later_floods) > 0 else pd.NaT,
            'beta_pre': beta_pre,
            'beta_post': beta_post,
            'delta_beta': delta_beta,
            'sediment_trend_benefit': delta_beta > 0,
            'sediment_trend_score': score,
            'post_window_months': len(post)
        })

    event_df = pd.DataFrame(events)

    if event_df.empty:
        return {
            'B_sediment_trend': np.nan,
            'N_dam_events': len(dam_events),
            'N_valid_events': 0,
            'event_table': event_df
        }

    return {
        'B_sediment_trend': event_df['sediment_trend_score'].mean(),
        'N_dam_events': len(dam_events),
        'N_valid_events': len(event_df),
        'event_table': event_df
    }


In [9]:
def compute_flood_regulation_failure(
    df,
    z_p_thr=1,
    z_wa_thr=1,
    z_gpp_thr=-1,
    z_q_thr=0,
    lag_months=1,
    use_gpp=True,   # True = strict ag-impact; False = hydrologic-only
):
    """
    Flood Regulation Failure (F_reg): fraction of extreme-precipitation events
    that still produce downstream flooding (and optionally crop stress).

    Returns
    -------
    df_z : DataFrame
        Input df with Z-scores + flags added.
    result : dict
        {
          'F_reg': float,
          'N_precip_events': int,
          'N_failed_events': int,
          'N_failed_and_highQ': int,
          'precip_event_table': DataFrame
        }
    """

    if df is None or df.empty:
        raise ValueError("Input dataframe is empty – cannot compute F_reg.")

    df = df.copy().sort_index()

    # ---- Z-scores (NaNs propagate safely)
    df['Z_WA']   = monthly_zscore(df['water_area_km2'])
    df['Z_GPP']  = monthly_zscore(df['modis_gpp'])
    df['Z_P']    = monthly_zscore(df['precip'])
    df['Z_Qout'] = monthly_zscore(df['outflow (m3/d)'])

    # ---- precip forcing months
    df['precip_extreme'] = df['Z_P'] >= z_p_thr

    # ---- downstream impact months (two modes)
    if use_gpp:
        df['impact_month'] = (df['Z_WA'] >= z_wa_thr) & (df['Z_GPP'] <= z_gpp_thr)
    else:
        df['impact_month'] = (df['Z_WA'] >= z_wa_thr)

    # ---- group precip months into events
    df['p_event_start'] = df['precip_extreme'] & (~df['precip_extreme'].shift(1).fillna(False))
    df['p_event_id'] = df['p_event_start'].cumsum()

    precip_df = df[df['precip_extreme']].copy()
    if precip_df.empty:
        return df, {
            'F_reg': 0.0,
            'N_precip_events': 0,
            'N_failed_events': 0,
            'N_failed_and_highQ': 0,
            'precip_event_table': pd.DataFrame()
        }

    events = []
    for eid, g in precip_df.groupby('p_event_id'):
        event_months = g.index

        # window = event months + lag months AFTER the event
        window_months = list(event_months)
        for l in range(1, lag_months + 1):
            window_months.extend(event_months + pd.DateOffset(months=l))

        window_months = pd.DatetimeIndex(window_months).intersection(df.index)

        # event diagnostics
        ZP_max  = df.loc[event_months, 'Z_P'].max()
        ZWA_max = df.loc[window_months, 'Z_WA'].max()
        ZGPP_min = df.loc[window_months, 'Z_GPP'].min()
        ZQ_max  = df.loc[window_months, 'Z_Qout'].max()

        failed = df.loc[window_months, 'impact_month'].any()
        failed_and_highQ = bool(failed and (ZQ_max >= z_q_thr))

        events.append({
            'event_id': int(eid),
            'start_date': event_months.min(),
            'end_date': event_months.max(),
            'duration_months': len(event_months),
            'ZP_max': float(ZP_max),
            'ZWA_max_window': float(ZWA_max),
            'ZGPP_min_window': float(ZGPP_min),
            'ZQ_max_window': float(ZQ_max),
            'failed_regulation': bool(failed),
            'failed_and_highQ': bool(failed_and_highQ),
        })

    event_df = pd.DataFrame(events)

    Np = len(event_df)
    Nfail = int(event_df['failed_regulation'].sum())
    Nfail_highQ = int(event_df['failed_and_highQ'].sum())

    F_reg = Nfail / Np if Np > 0 else 0.0

    return df, {
        'F_reg': F_reg,
        'N_precip_events': Np,
        'N_failed_events': Nfail,
        'N_failed_and_highQ': Nfail_highQ,
        'precip_event_table': event_df
    }


In [10]:
def compute_dam_attributed_flooding(
    df,
    z_wa_thr=1,
    z_gpp_thr=-1,
    z_q_thr=1,
    z_p_thr=1.0,
    lag_months=1
):
    """
    Compute Dam-Attributed Flood Fraction (F_dam) for a single site.

    Parameters
    ----------
    df : pandas.DataFrame
        Monthly dataframe indexed by datetime, containing:
        - water_area_km2
        - modis_gpp
        - precip
        - outflow (m3/d)
    z_*_thr : float
        Z-score thresholds
    lag_months : int
        Max lag (months) allowed for attribution

    Returns
    -------
    result : dict
        {
          'F_dam': float,
          'N_flood_events': int,
          'N_dam_attributed': int,
          'event_table': pandas.DataFrame
        }
    """

    df = df.copy().sort_index()

    # --------------------------------------------------
    # 1. Compute anomalies (NaNs propagate safely)
    # --------------------------------------------------
    df['Z_WA']   = monthly_zscore(df['water_area_km2'])
    df['Z_GPP']  = monthly_zscore(df['modis_gpp'])
    df['Z_P']    = monthly_zscore(df['precip'])
    df['Z_Qout'] = monthly_zscore(df['outflow (m3/d)'])

    # --------------------------------------------------
    # 2. Detect flood-impact months
    # --------------------------------------------------
    df['flood_month'] = (
        (df['Z_WA']  >= z_wa_thr) &
        (df['Z_GPP'] <= z_gpp_thr)
    )

    # --------------------------------------------------
    # 3. Group flood months into events
    # --------------------------------------------------
    df['event_start'] = (
        df['flood_month'] &
        (~df['flood_month'].shift(1).fillna(False))
    )

    df['event_id'] = df['event_start'].cumsum()
    flood_df = df[df['flood_month']].copy()

    if flood_df.empty:
        return df, {
            'F_dam': 0.0,
            'N_flood_events': 0,
            'N_dam_attributed': 0,
            'event_table': pd.DataFrame()
        }

    # --------------------------------------------------
    # 4. Event-level attribution (allow lag)
    # --------------------------------------------------
    events = []

    for eid, g in flood_df.groupby('event_id'):
        event_months = g.index

        # include lag window
        lagged_months = []
        for l in range(lag_months + 1):
            lagged_months.extend(event_months - pd.DateOffset(months=l))

        lagged_months = pd.DatetimeIndex(lagged_months).intersection(df.index)

        ZQ_max = df.loc[lagged_months, 'Z_Qout'].max()
        ZP_max = df.loc[lagged_months, 'Z_P'].max()

        dam_attributed = (
            (ZQ_max >= z_q_thr) &
            (ZP_max <  z_p_thr)
        )

        events.append({
            'event_id': eid,
            'start_date': event_months.min(),
            'end_date': event_months.max(),
            'duration_months': len(event_months),
            'ZQ_max': ZQ_max,
            'ZP_max': ZP_max,
            'dam_attributed': dam_attributed
        })

    event_df = pd.DataFrame(events)

    # --------------------------------------------------
    # 5. Compute F_dam
    # --------------------------------------------------
    N_f = len(event_df)
    N_dam = event_df['dam_attributed'].sum()

    F_dam = N_dam / N_f if N_f > 0 else 0.0

    result = {
    'F_dam': F_dam,
    'N_flood_events': N_f,
    'N_dam_attributed': int(N_dam),
    'event_table': event_df
    }

    return df, result


In [11]:
def compute_dam_supported_gpp_seasonal(
    df,
    gpp_quantile_thr=0.5,
    min_seasons=5,
    r2_gain_thr=0.05
):
    """
    Compute Dam-Supported Growing Season GPP Benefit (P_gpp)

    Question:
    Does dam outflow explain positive GPP anomalies better than precipitation alone
    during growing seasons?

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain (monthly, indexed by datetime):
        - Z_GPP
        - Z_Qout
        - Z_P

    gpp_quantile_thr : float
        Quantile threshold to define growing season months (default = 0.5)

    min_seasons : int
        Minimum number of growing seasons required for reliability

    r2_gain_thr : float
        Minimum R² improvement required to credit dam support

    Returns
    -------
    result : dict
        {
          'P_gpp': float,
          'N_seasons': int,
          'N_supported_seasons': int,
          'event_table': pandas.DataFrame
        }
    """

    df = df.copy().sort_index()

    # --------------------------------------------------
    # 1. Identify growing season months
    # --------------------------------------------------
    gpp_thr = df['modis_gpp'].quantile(gpp_quantile_thr)
    df['is_growing_season'] = df['modis_gpp'] >= gpp_thr

    # --------------------------------------------------
    # 2. Aggregate to growing-season years
    # --------------------------------------------------
    season_df = (
        df[df['is_growing_season']]
        .groupby(df[df['is_growing_season']].index.year)
        .agg({
            'Z_GPP': 'mean',
            'Z_Qout': 'mean',
            'Z_P': 'mean'
        })
        .dropna()
    )


    if len(season_df) < min_seasons:
        return {
            'P_gpp': np.nan,
            'N_seasons': len(season_df),
            'N_supported_seasons': 0,
            'event_table': pd.DataFrame()
        }

    # --------------------------------------------------
    # 3. Fit competing regression models
    # --------------------------------------------------
    X_p = season_df[['Z_P']].values
    X_pq = season_df[['Z_P', 'Z_Qout']].values
    y = season_df['Z_GPP'].values

    reg_p = LinearRegression().fit(X_p, y)
    reg_pq = LinearRegression().fit(X_pq, y)

    r2_p = reg_p.score(X_p, y)
    r2_pq = reg_pq.score(X_pq, y)

    beta_p = reg_p.coef_[0]
    beta_q = reg_pq.coef_[1]

    r2_gain = r2_pq - r2_p

    # --------------------------------------------------
    # 4. Identify dam-supported seasons
    # --------------------------------------------------
    season_df['dam_supported'] = (
        (season_df['Z_GPP'] > 0) &
        (season_df['Z_Qout'] > 0) &
        (beta_q > 0) &
        (r2_gain >= r2_gain_thr)
    )

    N_supported = season_df['dam_supported'].sum()
    N_total = len(season_df)

    P_gpp_raw = N_supported / N_total if N_total > 0 else np.nan

    # --------------------------------------------------
    # 5. Reliability weighting
    # --------------------------------------------------
    reliability = min(1.0, N_total / min_seasons)
    P_gpp = P_gpp_raw * reliability

    # --------------------------------------------------
    # 6. Assemble event table
    # --------------------------------------------------
    event_table = season_df.reset_index().rename(columns={
        'index': 'year'
    })

    return {
        'P_gpp': P_gpp,
        'N_seasons': N_total,
        'N_supported_seasons': int(N_supported),
        'beta_precip': beta_p,
        'beta_outflow': beta_q,
        'r2_precip': r2_p,
        'r2_precip_outflow': r2_pq,
        'r2_gain': r2_gain,
        'event_table': event_table
    }


In [12]:
def compute_growing_season_correlation_matrix(
    df,
    gpp_col="modis_gpp",
    quantile=0.5,
    cols=("Z_P", "Z_Qout", "Z_GPP"),
    min_obs=8
):
    """
    Correlation matrix restricted to growing-season months.
    """

    gpp_thr = df[gpp_col].quantile(quantile)
    gs = df[df[gpp_col] >= gpp_thr]

    sub = gs[list(cols)].dropna()

    if len(sub) < min_obs:
        return None

    return sub.corr()


In [13]:
def shrink_fraction(k, n, p0=0.1, m=2):
    """
    Empirical-Bayes shrinkage of a fraction k/n toward prior mean p0
    with prior strength m (pseudo-counts).
    """
    if n is None or n <= 0:
        return np.nan
    return (k + m*p0) / (n + m)

def compute_AIS_v2(
    # Metric 1
    N_dam_flood_events, N_flood_events,
    # Metric 2
    N_failed_reg_events, N_precip_events,
    # Metric 3
    N_positive_recovery, N_valid_recovery_events,
    # Metric 4
    N_supported_gpp_seasons, N_gpp_seasons,
    # Metric 5
    positive_regulation,
    # knobs
    prior_bad=0.2, prior_good=0.2, m_event=2, m_season=15,
    # w_dam=1, w_reg=0.55,
    # w_sed=0.55, w_pgpp=1,
    w_dam=0.65, w_reg=0.35,
    w_sed=0.25, w_pgpp=0.45, W_positive_regulation=0.3,
    return_components=True
):
    """
    Returns AIS_net in [-1,1] (approximately), plus optional components.
    """

    # ---- Convert to shrunk rates ----
    F_dam = shrink_fraction(N_dam_flood_events, N_flood_events, p0=prior_bad, m=m_event)
    F_reg = shrink_fraction(N_failed_reg_events, N_precip_events, p0=prior_bad, m=m_event)

    B_sed = shrink_fraction(N_positive_recovery, N_valid_recovery_events, p0=prior_good, m=m_event)
    P_gpp = shrink_fraction(N_supported_gpp_seasons, N_gpp_seasons, p0=prior_good, m=m_season)

    # If any are NaN (missing), treat as neutral contribution rather than breaking
    # Neutral for negative terms ~ prior_bad; for positive terms ~ prior_good
    if np.isnan(F_dam): F_dam = prior_bad
    if np.isnan(F_reg): F_reg = prior_bad
    if np.isnan(B_sed): B_sed = prior_good
    if np.isnan(P_gpp): P_gpp = prior_good

    # ---- Combine into negative and positive scores ----
    NIS = w_dam * F_dam + w_reg * F_reg          # higher worse
    PIS = w_sed * B_sed + w_pgpp * P_gpp + W_positive_regulation * positive_regulation        # higher better
    
    
    # ---- Net AIS ----
    W_pis = 1
    W_nis = 1.5
    AIS_net = W_pis * PIS - W_nis * NIS
    AIS_net = float(np.clip(AIS_net, -1, 1))

    if not return_components:
        return AIS_net

    return {
        "AIS_net": AIS_net,
        "NIS": float(NIS),
        "PIS": float(PIS),
        "F_dam_shrunk": float(F_dam),
        "F_reg_shrunk": float(F_reg),
        "B_sed_shrunk": float(B_sed),
        "P_gpp_shrunk": float(P_gpp),
    }


In [14]:
def add_scaled_ais(
    gdf,
    ais_col="AIS_net",
    out_col="AIS_net_scaled",
    method="robust",
    p_low=5,
    p_high=95
):
    """
    Add a scaled AIS column in the range [-1, 1].

    Parameters
    ----------
    gdf : GeoDataFrame or DataFrame
        Must contain `ais_col`.
    ais_col : str
        Column containing raw AIS values.
    out_col : str
        Name of the output scaled AIS column.
    method : {"minmax", "robust"}
        Scaling method:
        - "minmax": uses global min/max
        - "robust": uses percentile bounds (recommended)
    p_low, p_high : float
        Percentiles for robust scaling.

    Returns
    -------
    gdf : same object with new column added
    """

    x = gdf[ais_col].astype(float)

    if method == "minmax":
        lo = x.min()
        hi = x.max()

    elif method == "robust":
        lo = np.nanpercentile(x, p_low)
        hi = np.nanpercentile(x, p_high)

    else:
        raise ValueError("method must be 'minmax' or 'robust'")

    # Avoid division by zero
    if np.isclose(hi, lo):
        gdf[out_col] = np.nan
        return gdf

    # Scale to [0, 1]
    x_scaled = (x - lo) / (hi - lo)

    # Clip and rescale to [-1, 1]
    x_scaled = np.clip(x_scaled, 0, 1)
    gdf[out_col] = 2 * x_scaled - 1

    return gdf


In [15]:
import geopandas as gpd
import pandas as pd
import numpy as np
import traceback
from scipy.stats import theilslopes
from sklearn.linear_model import LinearRegression

import os

import warnings
warnings.filterwarnings("ignore")

In [ ]:
sites_list_file = '../Outputs/study_sites_v1_CA_stats.geojson'

gpp_water_area_path_format = '../Scripts/Core/revisions_GEN_Exam/outputs/revisions_gpp_ndvi/{}{}.csv'
downstream_river_fp = '../Outputs/DownstreamPaths'
combined_output_folder = '../Data/Grills_ratRun/rat_outputs/combined_outputs'
site_climate_path_format = combined_output_folder + '/catchment_climate/{}_{}.csv'
site_outflow_path_format = combined_output_folder + '/outflow/{}_{}.csv'

In [ ]:
grills_full_basins = gpd.read_file("../Outputs/basins_rat_sedi_v1.geojson")
grills_full_new = gpd.read_file("../Outputs/dams_rat_sedi_v1.geojson")
sites = gpd.read_file('../Outputs/study_sites_v1_CA_stats.geojson')
sites.drop_duplicates(subset=['DAM_NAME_left'], inplace=True)
sites.reset_index(drop=True, inplace=True)

sites_withBasins = determine_basin_name(sites, grills_full_basins, grills_full_new)

#drop duplicated
sites_withBasins = sites_withBasins.drop_duplicates(subset=['GRAND_ID'])
sites_withBasins['is_gpp_file_present'] = sites_withBasins.apply(lambda row: os.path.exists(gpp_water_area_path_format.format(row['GRAND_ID'], row['DAM_NAME_left'])), axis=1)
sites_withBasins['is_climate_file_present'] = sites_withBasins.apply(lambda row: os.path.exists(site_climate_path_format.format(row['GRILSS RID'], row['Reservoir'].replace(' ', '_'))), axis=1)
sites_withBasins['is_outflow_file_present'] = sites_withBasins.apply(lambda row: os.path.exists(site_outflow_path_format.format(row['GRILSS RID'], row['Reservoir'].replace(' ', '_'))), axis=1)
sites_withBasins['all_files_present'] = sites_withBasins.apply(lambda row: row['is_gpp_file_present'] and row['is_climate_file_present'] and row['is_outflow_file_present'], axis=1)
sites_withBasins.dropna(subset = ['GRILSS RID'], inplace = True)
sites_withBasins.head(2)

,GRAND_ID,RES_NAME,DAM_NAME_left,ALT_NAME,RIVER,ALT_RIVER,MAIN_BASIN,SUB_BASIN,NEAR_CITY,ALT_CITY,...,perc_grainNet_area_of_total_command_area,geometry,RIVER_BASIN,index_right,GRILSS RID,Reservoir,is_gpp_file_present,is_climate_file_present,is_outflow_file_present,all_files_present
0,4702,Tarbela,Tarbela,NaN,Indus,NaN,Indus,NaN,Haripur,NaN,...,81.711432,POINT (72.69110 34.09134),INDUS,653.0,654,Tarbela,False,True,True,False
1,4707,Mangla,Mangla,NaN,Jhelum,NaN,Indus,NaN,Jhelum,NaN,...,85.952095,POINT (73.64447 33.14397),INDUS,1078.0,10066,Mangla,False,True,True,False


## Analysis

In [19]:
counter = 1
counter_dam_floods = 0
missing_files_dict = {}

sites_withBasins = sites_withBasins[sites_withBasins['all_files_present'] == True]

for site in sites_withBasins.iterrows():
    
    site_name = site[1]['DAM_NAME_left']
    site_grills_name = site[1]['Reservoir']
    site_grandID = site[1]['GRAND_ID']
    site_basin = site[1]['MAIN_BASIN']
    site_grillsID = site[1]['GRILSS RID']
    
    print(f"Processing site {counter}: {site_name} | GRILSS ID: {site_grillsID} | GRAND ID: {site_grandID} | Basin: {site_basin}")
    
    #reading files
    site_gpp_and_waterArea_file = os.path.join(gpp_water_area_path_format.format(site_grandID, site_name))
    site_precip_file = os.path.join(site_climate_path_format.format(site_grillsID, site_grills_name.replace(' ', '_')))
    site_outflow_file = os.path.join(site_outflow_path_format.format(site_grillsID, site_grills_name.replace(' ', '_')))

    site_gpp_and_waterArea = pd.read_csv(site_gpp_and_waterArea_file)
    site_precip = pd.read_csv(site_precip_file)
    site_outflow = pd.read_csv(site_outflow_file)
    
    ##computing monthly values
    site_precip['date'] = pd.to_datetime(site_precip['time'])
    site_precip.set_index('date', inplace = True)
    site_precip_month = compute_monthly_metrics(site_precip[['precip']])

    site_precip_month['Year'] = site_precip_month.index.year
    site_precip_yearly = site_precip_month.groupby(by = ["Year"]).sum()
    site_precip_yearly_mean = site_precip_yearly['precip'].mean()
    
    site_outflow['date'] = pd.to_datetime(site_outflow['date'])
    site_outflow.set_index('date', inplace = True)
    site_outflow_month = compute_monthly_metrics(site_outflow[['outflow (m3/d)']])
    site_outflow_month['outflow (m3/s)'] = site_outflow_month['outflow (m3/d)'] / 86400
    
    downstream_river_gpd = gpd.read_file(f'{downstream_river_fp}/{site_grandID}_{site_name}_riverNet.geojson')
    downtream_river_length = downstream_river_gpd['LENGTH_KM'].sum()
    print(f'River Length = {downtream_river_length}')

    ######## data cleaning #########
    ################################
    
    #fixing null values in gpp using interpolation
    site_gpp_inter = site_gpp_and_waterArea.copy()

    site_gpp_inter['date'] = pd.to_datetime(site_gpp_inter['date'])
    site_gpp_inter = site_gpp_inter.sort_values('date').set_index('date')

    site_gpp_inter['water_area_km2'] = (
        site_gpp_inter['water_area_km2']
        .interpolate(method='time', limit_direction='both')
    )
    site_gpp_inter['landsat_cloud_pct'] = (
        site_gpp_inter['landsat_cloud_pct']
        .interpolate(method='time', limit_direction='both')
    )
    
    #fixing date issues in precip and outflow
    site_outflow_month = site_outflow_month.copy()
    site_outflow_month['date'] = [x + '-01' for x in site_outflow_month.index.astype(str)]
    site_outflow_month['date'] = pd.to_datetime(site_outflow_month['date'])
    
    site_precip_month = site_precip_month.copy()
    site_precip_month['date'] = [x + '-01' for x in site_precip_month.index.astype(str)]
    site_precip_month['date'] = pd.to_datetime(site_precip_month['date'])
    #set date and index
    site_outflow_month.set_index('date', inplace = True)
    site_precip_month.set_index('date', inplace = True)

    #creating merged gpp outflow precip data
    merged_gpp_outflow_precip = (
    site_gpp_inter
    .join(site_precip_month, how='inner')
    .join(site_outflow_month, how='inner')
    )

    #avoid extreme cloudy months. cloud prc > 70%
    CLOUD_THR = 80
    merged_gpp_outflow_precip.loc[merged_gpp_outflow_precip['landsat_cloud_pct'] > CLOUD_THR, 'water_area_km2'] = np.nan
    
    ############ end of data cleaning ############
    ##############################################
    
   
    
    ###### start of AIS metric computation #######
    ##############################################
    #add climatoglogical mean columns for precip, outlfow, gpp, water area
    merged_gpp_outflow_precip['clim_mean_precip'] = merged_gpp_outflow_precip['precip'].groupby(merged_gpp_outflow_precip.index.month).transform('mean')
    merged_gpp_outflow_precip['clim_mean_outflow'] = merged_gpp_outflow_precip['outflow (m3/d)'].groupby(merged_gpp_outflow_precip.index.month).transform('mean')
    merged_gpp_outflow_precip['clim_mean_gpp'] = merged_gpp_outflow_precip['modis_gpp'].groupby(merged_gpp_outflow_precip.index.month).transform('mean')
    merged_gpp_outflow_precip['clim_mean_water_area'] = merged_gpp_outflow_precip['water_area_km2'].groupby(merged_gpp_outflow_precip.index.month).transform('mean')

    
    ##anomalous precip months
    #tracks for each site how many months have anomalous precip defined as 1 std above mean for that month
    merged_gpp_outflow_precip['Z_Precip'] = monthly_zscore(merged_gpp_outflow_precip['precip'])
    merged_gpp_outflow_precip['Z_water'] = monthly_zscore(merged_gpp_outflow_precip['water_area_km2'])
    merged_gpp_outflow_precip['Z_gpp'] = monthly_zscore(merged_gpp_outflow_precip['modis_gpp'])
    merged_gpp_outflow_precip['Z_outflow'] = monthly_zscore(merged_gpp_outflow_precip['outflow (m3/d)'])

    #number of high precip months
    site_high_precip_months = len(merged_gpp_outflow_precip[merged_gpp_outflow_precip['Z_Precip'] >= 1.0])
    
    #number of high precip months when water area is also high
    site_high_precip_and_water_months = len(merged_gpp_outflow_precip[(merged_gpp_outflow_precip['Z_Precip'] >= 1.0) & (merged_gpp_outflow_precip['Z_water'] >= 1.0)])
    
    #succesful regulation months - high precip but no high outflow
    successful_regulation_months = len(merged_gpp_outflow_precip[(merged_gpp_outflow_precip['Z_Precip'] >= 1.0) & (merged_gpp_outflow_precip['Z_outflow'] < 0)])   
    
    ## Correlation Matrix ##
    
    corr_gs = compute_growing_season_correlation_matrix(
    merged_gpp_outflow_precip,
    gpp_col="modis_gpp",
    cols=("Z_Precip", "Z_outflow", "Z_gpp", "Z_water")
    )
    print(corr_gs)
    ## AIS Metric 1 - compute dam attributed flooding ##
    res_df, result = compute_dam_attributed_flooding(merged_gpp_outflow_precip)
    summary_df = pd.DataFrame([{
        'F_dam': result['F_dam'],
        'N_flood_events': result['N_flood_events'],
        'N_dam_attributed': result['N_dam_attributed']
    }])
    
    ### AIS Metric 2 - compute flood regulation failure
    df_z, reg = compute_flood_regulation_failure(merged_gpp_outflow_precip)

    summary_reg = pd.DataFrame([{
        'F_reg': reg['F_reg'],
        'N_precip_events': reg['N_precip_events'],
        'N_failed_events': reg['N_failed_events'],
        'N_failed_and_highQ': reg['N_failed_and_highQ'],
    }])
    
    ##AIS Metric 3 - Long Term Flooding Sedimentation Benefit
    number_of_flood_events = result['N_flood_events']
    if number_of_flood_events > 0:
        sediment_result = compute_sediment_trend_benefit(
            df=res_df,
            flood_event_table=result['event_table']
            )
    else:
        sediment_result = {
            'B_sediment_trend': np.nan,
            'N_dam_events': 0,
            'N_valid_events': 0,
            'event_table': pd.DataFrame()
        }
        
    B_sed = sediment_result.get('B_sediment_trend', 0.0)
    N_dam_recovery_events = sediment_result.get('N_dam_events', 0)
    N_valid_recovery_events = sediment_result.get('N_valid_events', 0)
    if sediment_result.get('event_table') is not None and not sediment_result['event_table'].empty:
        N_positive_recovery = sediment_result['event_table']['sediment_trend_benefit'].sum()
    else:
        N_positive_recovery = 0
    positive_recovers_perc = N_positive_recovery / N_valid_recovery_events if N_valid_recovery_events > 0 else np.nan
    ## AIS Metric 4 - Growing Season Dam Benefit to GPP
    gpp_benefit_result = compute_dam_supported_gpp_seasonal(df=res_df)
    N_gpp_seasons = gpp_benefit_result.get("N_seasons", 0)
    N_supported_gpp_seasons = gpp_benefit_result.get("N_supported_seasons", 0)
    
    ##regulation postive
    positive_regulation = successful_regulation_months / site_high_precip_months if site_high_precip_months > 0 else np.nan
    ## Computing AIS ##
    N_flood_events = result['N_flood_events']
    N_dam_flood_events = result['N_dam_attributed']
    N_precip_events = reg['N_precip_events']
    N_failed_reg_events = reg['N_failed_events']
    N_postive_reg_events = successful_regulation_months
    
    ais_pack = compute_AIS_v2(
        N_dam_flood_events=N_dam_flood_events,
        N_flood_events=N_flood_events,
        N_failed_reg_events=N_failed_reg_events,
        N_precip_events=N_precip_events,
        N_positive_recovery=N_positive_recovery,
        N_valid_recovery_events=N_valid_recovery_events,
        N_supported_gpp_seasons=N_supported_gpp_seasons,
        N_gpp_seasons= N_gpp_seasons,
        positive_regulation=positive_regulation
    )
    
  
    ##### adding data to site dataframe #####
    #########################################
    idx = site[0]
    
    #saving precip mean, river length
    sites_withBasins.at[idx, 'precip_annual_mean'] = int(site_precip_yearly_mean)
    sites_withBasins.at[idx, 'downstream_river_length'] = int(downtream_river_length)

    
    sites_withBasins.at[idx, 'N_high_precip_months'] = int(site_high_precip_months)
    sites_withBasins.at[idx, 'N_high_precip_and_water_months'] = int(site_high_precip_and_water_months)
    sites_withBasins.at[idx, 'N_flood_events'] = result['N_flood_events']
    sites_withBasins.at[idx, 'N_flood_events'] = result['N_flood_events']
    sites_withBasins.at[idx, 'N_dam_attributed'] = result['N_dam_attributed']
    sites_withBasins.at[idx, 'F_dam'] = result['F_dam']
    sites_withBasins.at[idx, 'F_reg'] = reg['F_reg']
    sites_withBasins.at[idx, 'N_failed_events'] = reg['N_failed_events']
    sites_withBasins.at[idx, 'N_failed_and_highQ'] = reg['N_failed_and_highQ']
    sites_withBasins.at[idx, 'N_successful_regulation_months'] = successful_regulation_months
    sites_withBasins.at[idx, 'N_positive_recovery'] = N_positive_recovery
    sites_withBasins.at[idx, 'B_sed'] = positive_recovers_perc

    sites_withBasins.at[idx, 'P_gpp'] = gpp_benefit_result.get("P_gpp", np.nan)
    sites_withBasins.at[idx, 'N_gpp_seasons'] = N_gpp_seasons
    sites_withBasins.at[idx, 'N_supported_gpp_seasons'] = N_supported_gpp_seasons
    sites_withBasins.at[idx, 'Positive_regulation'] = positive_regulation

    sites_withBasins.at[idx, 'AIS_net'] = ais_pack['AIS_net']
    sites_withBasins.at[idx, 'NIS'] = ais_pack['NIS']
    sites_withBasins.at[idx, 'PIS'] = ais_pack['PIS']

    sites_withBasins.at[idx, 'correl_gpp_outflow_growing'] = corr_gs['Z_gpp'][1] if corr_gs is not None else np.nan
    sites_withBasins.at[idx, 'correl_gpp_precip_growing'] = corr_gs['Z_gpp'][0] if corr_gs is not None else np.nan
    sites_withBasins.at[idx, 'correl_water_precip_growing'] = corr_gs['Z_water'][0] if corr_gs is not None else np.nan
    sites_withBasins.at[idx, 'correl_water_outflow_growing'] = corr_gs['Z_water'][1] if corr_gs is not None else np.nan


    
    
    counter += 1

sites_withBasins['Regulation%'] = sites_withBasins['N_successful_regulation_months'] / sites_withBasins['N_high_precip_months'] * 100
sites_withBasins = add_scaled_ais(
    sites_withBasins,
    method="minmax"
)
    

Processing site 1: Salal | GRILSS ID: 427 | GRAND ID: 4708 | Basin: Indus
River Length = 647.1799999999998
           Z_Precip  Z_outflow     Z_gpp   Z_water
Z_Precip   1.000000   0.331751  0.313455  0.124942
Z_outflow  0.331751   1.000000  0.331539  0.159315
Z_gpp      0.313455   0.331539  1.000000  0.002996
Z_water    0.124942   0.159315  0.002996  1.000000
Processing site 2: Kalri Lake | GRILSS ID: 10230 | GRAND ID: 4725 | Basin: nan
River Length = 153.79000000000002
           Z_Precip  Z_outflow     Z_gpp   Z_water
Z_Precip   1.000000   0.105668  0.492523  0.037390
Z_outflow  0.105668   1.000000  0.543709  0.296500
Z_gpp      0.492523   0.543709  1.000000  0.212594
Z_water    0.037390   0.296500  0.212594  1.000000
Processing site 3: Sardar Sarovar | GRILSS ID: 10061 | GRAND ID: 4734 | Basin: Narmada
River Length = 145.45000000000002
           Z_Precip  Z_outflow     Z_gpp   Z_water
Z_Precip   1.000000   0.373986  0.309437  0.237557
Z_outflow  0.373986   1.000000  0.415430  0.263